In [ ]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 0.内存优化
把默认的数据类型缩短到最小，方便特征工程。

In [ ]:
df=pd.read_csv('../data/credit_data(1).csv')
df.head()

def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.
    """
    start_mem = df.memory_usage(deep=True).sum()/1024**2
    print('内存占用{:.2f} MB'.format(start_mem))

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_integer_dtype(col_type):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            elif pd.api.types.is_float_dtype(col_type) and df[col].notna().any():
                original_values = df[col].to_numpy()
                for candidate_type in (np.float16, np.float32):
                    if c_min > np.finfo(candidate_type).min and c_max < np.finfo(candidate_type).max:
                        candidate = df[col].astype(candidate_type)
                        restored_values = candidate.astype(col_type).to_numpy()
                        if np.array_equal(original_values, restored_values, equal_nan=True):
                            df[col] = candidate
                            break

    end_mem = df.memory_usage(deep=True).sum()/1024**2
    print('优化后内存为: {:.2f} MB'.format(end_mem))
    print('内存使用减少 {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

df=reduce_mem_usage(df)

## 1.EDA
- 目的：验证通过业务逻辑提出的猜想，并发现无法通过业务逻辑得到的数据关系，为后续的特征工程做准备
- 动作：清洗数据 & 可视化（针对单个变量统计，针对多个变量研究关系）& 提出假设
- 主要流程： 浏览数据->数据类型转变->剔除重复值->填补缺失值->清洗异常值->目标样本分析概览->数据关系概览

### 1.1数据初步整理
先初步得到**数据形状、数据类型。**
根据实际意义和真实取值改变数据类型。
将数据类型分为数值型和类别型变量，进一步输出数值型变量的观测值分布，发现:
1. policyCode只有1个取值，无用，后续可考虑删除。
2. 部分变量呈右偏分布，存在较大的尾部取值，需结合业务规则与变量间关系判断，不能直接视为错误值。

In [ ]:
df.shape
df.info()

# id 只用于数据匹配，不作为模型特征
df["id"] = df["id"].astype("string")

# 按照信用等级顺序编码，避免 factorize 按出现顺序编码
grade_order = list("ABCDEFG")
subgrade_order = [f"{grade}{level}" for grade in grade_order for level in range(1, 6)]
grade_codes = df["grade"].map({value: index for index, value in enumerate(grade_order)})
subgrade_codes = df["subGrade"].map({value: index for index, value in enumerate(subgrade_order)})
# if (df["grade"].notna() & grade_codes.isna()).any():
#     bad_values = df.loc[df["grade"].notna() & grade_codes.isna(), "grade"].unique().tolist()
#     raise ValueError(f"grade 中存在未定义的取值: {bad_values}")
# if (df["subGrade"].notna() & subgrade_codes.isna()).any():
#     bad_values = df.loc[df["subGrade"].notna() & subgrade_codes.isna(), "subGrade"].unique().tolist()
#     raise ValueError(f"subGrade 中存在未定义的取值: {bad_values}")
df["grade"] = grade_codes.astype("Int8")
df["subGrade"] = subgrade_codes.astype("Int8")

# 将工龄统一转换为可空整数，异常非空值显式报错
employment_length = df["employmentLength"].astype("string").str.strip()
employment_length = employment_length.replace({"10+ years": "10 years", "< 1 year": "0 years"})
employment_years = employment_length.str.extract(r"^(\d+)\s*years?$", expand=False)
invalid_employment_length = employment_length.notna() & employment_years.isna()
# if invalid_employment_length.any():
#     invalid_values = employment_length[invalid_employment_length].unique().tolist()
#     raise ValueError(f"无法解析的 employmentLength 取值: {invalid_values}")
df["employmentLength"] = pd.to_numeric(employment_years, errors="raise").astype("Int8")

# 明确日期格式；最早信用额度只精确到月份，转成时间戳后以每月1日作为锚点
df["issueDate"] = pd.to_datetime(df["issueDate"], format="%Y-%m-%d", errors="raise")
df["earliesCreditLine"] = pd.to_datetime(df["earliesCreditLine"], format="%b-%Y", errors="raise")

# 仅对实际含义为整数的字段转换；连续变量保持浮点型
integer_cols = [
    "employmentTitle", "postCode", "delinquency_2years", "ficoRangeLow", "ficoRangeHigh",
    "openAcc", "pubRec", "pubRecBankruptcies", "totalAcc", "title", "policyCode",
    *[f"n{i}" for i in range(15)],
]
for col in integer_cols:
    values = df[col].dropna()
    if not values.mod(1).eq(0).all():
        raise ValueError(f"{col} 含有非整数值，不能转换为整数")
    df[col] = df[col].astype("Int64")

df.info()

In [ ]:
#数据分成类别和数值
numeric_cols = [
    col for col in df.columns
    if pd.api.types.is_numeric_dtype(df[col])
]

categorical_cols = [
    col for col in df.columns
    if pd.api.types.is_object_dtype(df[col])
]

#输出数值型变量分布
# f,ax=plt.subplots((len(numeric_cols)+3)//4,4,figsize=(12,5*len(numeric_cols)//4))
# for i, col in enumerate(numeric_cols):
#     sns.histplot(df[col], ax=ax[i // 4, i % 4],bins=40)
n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

# 调大整体画布高度，宽松一点
f, ax = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))

# 把ax展平成一维数组，索引更简单，避免 i//4,i%4
ax = ax.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], ax=ax[i], bins=40)
    ax[i].set_title(col, fontsize=9)    # 标题缩小
    ax[i].tick_params(labelsize=8)     # 刻度字缩小

# 多余的空坐标轴关掉（如果总数不是4的倍数，最后会有空子图）
for j in range(len(numeric_cols), len(ax)):
    ax[j].axis("off")

# 核心：自动调整布局，防止文字重叠
plt.tight_layout()
plt.show()

In [ ]:
duplicated_count=df.duplicated(subset=[col for col in df.columns if col != "id"]).sum()
print(f"重复值有{duplicated_count}个")

### 1.3 缺失值检查
输出缺失率，发现多组变量缺失率相同，结合数据分布，猜测缺失的观测值相同，逐组检查。
验证得到n4 和n10 缺失的观测值相同，n*组（除n4 n10 n11）外缺失的观测值相同。

**缺失可分为MCAR MAR MNAR**，信贷分析中，数据缺失很可能是客户不愿披露隐私数据、刻意隐瞒，这也是一种特征，因此暂时不填充。

In [ ]:
missing_data=df.isna()
missing_percentage=missing_data.sum()/len(df)
missing_percentage[missing_percentage > 0].sort_values().plot(kind="bar").set_title("missing_percentage")

In [ ]:
missing_n4=df["n4"].isna()
missing_n10=df["n10"].isna()
if missing_n4.equals(missing_n10):
    print("n4 & n10 为相同的缺失值")
# df["n4"] = df["n4"].replace(r"\N", -999, regex=False).fillna(-999)
# df["n10"] = df["n10"].replace(r"\N", -999, regex=False).fillna(-999)

missing_n1=df["n1"].isna()
missing_col=[]
for col in df.columns:
    if df[col].isna().equals(missing_n1):
        missing_col.append(col)
print(f"与 n1 为相同的缺失值的列: {missing_col}")
# for col in missing_col:
#     df[col] = df[col].replace(r"\N", -999, regex=False).fillna(-999)

### 1.4 异常值处理
先处理逻辑上的硬性错误，再根据数据分布情况判断真实情况。
检查负值、IQR极端值、无穷值及利用率是否超过100，并输出字段最小值、最大值和异常数量。

In [ ]:
continuous_columns = ["annualIncome", "dti", "revolBal", "revolUtil", "loanAmnt", "installment", "interestRate"]

def check_anomalies(data):
    negative_cols = ["annualIncome", "loanAmnt", "dti", "openAcc", "totalAcc", "pubRec", "pubRecBankruptcies", "delinquency_2years"]
    business_flags = pd.DataFrame(index=data.index)
    rows = []
    for col in dict.fromkeys(negative_cols + continuous_columns):
        values = data[col].replace([np.inf, -np.inf], np.nan).dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        business_flags[f"{col}_negative"] = data[col] < 0
        business_flags[f"{col}_infinite"] = np.isinf(data[col])
        business_flags[f"{col}_iqr_low"] = data[col].replace([np.inf, -np.inf], np.nan) < low
        business_flags[f"{col}_iqr_high"] = data[col].replace([np.inf, -np.inf], np.nan) > high
        rows.append({
            "字段": col, "最小值": values.min(), "最大值": values.max(),
            "最大值大于最小值": values.max() > values.min(),
            "负值数": int(business_flags[f"{col}_negative"].sum()),
            "极小值数": int(business_flags[f"{col}_iqr_low"].sum()),
            "极大值数": int(business_flags[f"{col}_iqr_high"].sum()),
            "无穷值数": int(business_flags[f"{col}_infinite"].sum()),
            "超过100数": int((data[col] > 100).sum()) if col == "revolUtil" else 0,
        })
    business_flags["dti_999"] = data["dti"] == 999
    business_flags["revolUtil_above_100"] = data["revolUtil"] > 100
    return pd.DataFrame(rows), business_flags

business_summary, business_flags = check_anomalies(df)
display(business_summary)

#### 1.4.2 在副本中处理明确无效的取值
df_clean 保留所有样本，只将负债收入比为负的字段置为缺失，并将上述连续变量中的正负无穷置为缺失；原始值仍在 df 中，原因仍在 business_flags 中。其他命中规则暂不自动修改。

DTI_999_IS_MISSING 默认 False；只有确认999是缺失编码后才改为 True。零收入、利用率超过100以及真实高收入、高余额仍保留。后续建模如需使用清洗结果，应从 df_clean 构造特征，而不是重新使用原始 df。

In [ ]:
DTI_999_IS_MISSING = False

df_clean = df.copy()
for column in continuous_columns:
    df_clean[column] = df_clean[column].mask(business_flags[f"{column}_infinite"].eq(1))

invalid_dti = business_flags["dti_negative"].eq(1)
if DTI_999_IS_MISSING:
    invalid_dti = invalid_dti | business_flags["dti_999"].eq(1)
df_clean["dti"] = df_clean["dti"].mask(invalid_dti)

cleaning_summary = pd.DataFrame({
    "处理前缺失数": df[continuous_columns].isna().sum(),
    "处理后缺失数": df_clean[continuous_columns].isna().sum(),
})
cleaning_summary["新增缺失数"] = cleaning_summary["处理后缺失数"] - cleaning_summary["处理前缺失数"]
display(cleaning_summary)
assert df_clean.index.equals(df.index)
assert df_clean.shape == df.shape

### 1.5 相关性分析
输出所有数值变量的相关系数热力图，发现：
1. 分期付款金额(installment)与贷款金额(loanAmnt)的相关性最高，达到0.93，合理。因为贷款金额越大，分期付款金额也越大。
2. 大部分匿名变量(n1~n10)相关系数很高，结合上面的缺失值分析可知，大概率是同一系列行为的标签。方便后面聚类。
3. 借款人信用档案中未结信用额度的数量(openAcc)与n10高度相关，推测匿名变量为过去的信用卡行为。


In [ ]:
f,ax=plt.subplots(1,1,figsize=(12, 12))
corr=df[numeric_cols].corr()
mask_annot = np.where(np.abs(corr) > 0.6, corr.round(2), "")
sns.heatmap(corr, annot=mask_annot,annot_kws={"size":6},linewidths=0.1,linecolor="white", fmt="",ax=ax, cmap="coolwarm",cbar_kws={"shrink":0.8})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(data=df, x="installment", y="loanAmnt", ax=ax1)
sns.scatterplot(data=df, x="openAcc", y="n10", ax=ax2)

plt.tight_layout()
plt.show()


### 1.6目标样本分类
违约记录明显少于不违约记录，正负样本分布大致相同。

In [ ]:
plt.hist(df["isDefault"])
plt.title("target_sample")
plt.show

df_positve = df[df['isDefault'] == 1]
df_negative = df[df['isDefault'] != 1]
f, ax = plt.subplots(len(numeric_cols),2,figsize = (10,80))
for i,col in enumerate(numeric_cols):
    sns.histplot(df_positve[col],ax = ax[i,0],color = "blue",bins=20)
    sns.histplot(df_negative[col],ax = ax[i,1],color = 'red',bins=20)
ax[0,0].set_title("positive")
ax[0,1].set_title("negative")
plt.subplots_adjust(hspace = 1)
